### 在上一节课中，总结
- 将训练好的神经网络模型，保存为savemodel模式，读取其中的关键输入输出信息
- 拉取对应的tensorflow serving镜像 docker pull docker.m.daocloud.io/tensorflow/serving:2.13.1
- 将savemodel的路径，挂载到镜像的容器中去
- 将下载的图像，通过创建一个gRPC通道到服务器，处理后，传送给容器，容器来负责推理和出结果

In [3]:
#  将上一节课的代码内容，清理一下，写个ipynb文件，然后将其转换为py脚本。
#  整理完的ipynb文件，在11_k8s目录中
!jupyter nbconvert --to script tf-serving-connect.ipynb

[NbConvertApp] Converting notebook tf-serving-connect.ipynb to script
[NbConvertApp] Writing 1507 bytes to tf-serving-connect.py


In [3]:
%run tf-serving-connect.py

{'dress': -2.1015233993530273, 'hat': -5.5352373123168945, 'longsleeve': -2.531212329864502, 'outwear': -1.957198977470398, 'pants': 9.972639083862305, 'shirt': -1.3964393138885498, 'shoes': -2.5511271953582764, 'shorts': -0.28122517466545105, 'skirt': -3.9467363357543945, 't-shirt': -3.624239206314087}


In [ ]:
# 上面测试了python脚本的结果，与第1节克重的notebook中的结果一致
# 下面我们给这个脚本加上RESTFUL，做成一个web服务模式，就是前一节课中，第一幅图中画的：gateway
使用gunicorn，gunicorn 库安装到uv环境中
然后启动web服务：  
gunicorn --bind 0.0.0.0:9696 02_tf_serving_conect_gateway:app
最后写测试脚本，进行测试：02_test_gateway.py
import requests

url = 'http://127.0.0.1:9696/predict'
image_url = {"url": "https://pic.mksucai.com/00/10/42/60e8c5905c7695a3.webp"}

response = requests.post(url, json=image_url)      

print("状态码:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("响应体:")
print(response.text)


## 去除 TensorFlow 依赖

***注意库大小***：
- TensorFlow本身大约是1.7GB， 仅 CPU 版本的内存仍约为 400 MB。
- 我们没有 想在我们的门户里有这么重的依赖。
- 在我们的笔记本里， 只用到TensorFlow 中的一个函数，即 将我们的numpy数组转换为protobuf格式。拖着整个 光是为了这个就和我们图书馆一起，实在太过分了。
- 把下面的脚本，写入到 tf_serving_conect_gateway_tensorproto 这个新的网关脚本中,修改：app = Flask('gateway_tensorproto')

In [ ]:
from tensorflow.core.framework import tensor_pb2, tensor_shape_pb2, types_pb2


def dtypes_as_dtype(dtype):
    if dtype == "float32":
        return types_pb2.DT_FLOAT
    raise Exception("dtype %s is not supported" % dtype)


def make_tensor_proto(data):
    shape = data.shape
    dims = [tensor_shape_pb2.TensorShapeProto.Dim(size=i) for i in shape]
    proto_shape = tensor_shape_pb2.TensorShapeProto(dim=dims)

    proto_dtype = dtypes_as_dtype(data.dtype)

    tensor_proto = tensor_pb2.TensorProto(dtype=proto_dtype, tensor_shape=proto_shape)
    tensor_proto.tensor_content = data.tostring()

    return tensor_proto


def np_to_protobuf(data):
    if data.dtype != "float32":
        data = data.astype("float32")
    return make_tensor_proto(data)

In [ ]:
# 修改完，测试：启动容器，启动：gunicorn --bind 0.0.0.0:9696 tf_serving_conect_gateway_tensorproto:app 

### 这一步做完了，测试成功后，看这个：from tensorflow.core.framework import tensor_pb2, tensor_shape_pb2, types_pb2 
### 这些必要的类，都需要依赖tensorflow，我们需要构建这些类，使用工具脚本，进行构造，脚本如下：
### 注意，如果是wsl环境，建议单独下载后，在windows解压，wsl太慢了，看代码，注意解压的目录正确。就是我的11_k8s中运行这个脚本。单独下载，单独解压，然后再执行这个脚本。

In [ ]:
# 出错就退出
set -e

# 设置版本号变量
TF_VERSION=${TF_VERSION:-2.13.1}
TFS_VERSION=${TFS_VERSION:-${TF_VERSION}}

# 打印当前使用的版本，方便调试。
echo "running tf-serving-proto.sh with TF_VERSION=${TF_VERSION} and TFS_VERSION=${TFS_VERSION}"

# 安装 grpcio-tools
uv pip install grpcio-tools


wget https://ghproxy.net://https://github.com/tensorflow/tensorflow/archive/v${TF_VERSION}.zip -O tf.zip
unzip tf.zip && rm tf.zip

wget https://ghproxy.net://https://github.com/tensorflow/serving/archive/${TFS_VERSION}.zip -O tf-serving.zip
unzip tf-serving.zip && rm tf-serving.zip

# 把 Serving 的 tensorflow_serving 目录挪进 TF 源码目录里。
mv serving-${TFS_VERSION}/tensorflow_serving tensorflow-${TF_VERSION}

mkdir tf_temporary_folder

cd tensorflow-${TF_VERSION}

TF_SERVING_PROTO=../tf_temporary_folder

python -m grpc.tools.protoc \
    ./tensorflow/core/framework/*.proto \
    --python_out=${TF_SERVING_PROTO} \
    --grpc_python_out=${TF_SERVING_PROTO} \
    --proto_path=.

python -m grpc.tools.protoc \
    ./tensorflow/core/example/*.proto \
    --python_out=${TF_SERVING_PROTO} \
    --grpc_python_out=${TF_SERVING_PROTO} \
    --proto_path=.

python -m grpc.tools.protoc \
    ./tensorflow/core/protobuf/*.proto \
    --python_out=${TF_SERVING_PROTO} \
    --grpc_python_out=${TF_SERVING_PROTO} \
    --proto_path=.

python -m grpc.tools.protoc \
    ./tensorflow_serving/apis/*.proto \
    --python_out=${TF_SERVING_PROTO} \
    --grpc_python_out=${TF_SERVING_PROTO} \
    --proto_path=.

mv LICENSE ..


cd ..

rm -r serving-${TFS_VERSION} tensorflow-${TF_VERSION}
mv tf_temporary_folder/* .

rm -fr tf_temporary_folder

touch tensorflow/__init__.py
touch tensorflow_serving/__init__.py

In [ ]:
# 做完后，测试：启动容器，启动：gunicorn --bind 0.0.0.0:9696 tf_serving_conect_gateway_tensorproto:app 